In [ ]:
from huggingface_hub import hf_hub_download
import json

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
TEST_FILE = "Phase_1_Pretraining/Test/test.json"
HF_TOKEN = "enter_your_own"

test_file_path = hf_hub_download(repo_id=HF_DATASET_REPO, filename=TEST_FILE, repo_type="dataset", token=HF_TOKEN)
with open(test_file_path, "r", encoding="utf-8") as f:
    test_data = json.load(f)

In [ ]:
from bpetokenizer import *
tokenizer = load_bpe_tokenizer()

In [ ]:
from model_arch import *

device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")

cfg = LMConfig(vocab_size=50260)
model = DecoderLM(cfg)
model.load_state_dict(torch.load("checkpoint.pt", map_location=device, weights_only=True))
model.to(device)
model.eval()

In [ ]:
import torch.nn.functional as F

max_tokens = 250
temp = 0.8
topp = 0.9
repetition_penalty = 1.1

def continue_prompt(prompt):
    tokens = tokenizer.encode(prompt)[max_tokens-1024:]
    idx = torch.tensor(tokens, device=device).unsqueeze(0)
    generated = []

    with torch.inference_mode():
        for i in range(max_tokens):
            out = model(idx)
            next_token_logits=out[:, -1, :]/temp

            for t in set(generated[-64:]):
                next_token_logits[0, t] /= repetition_penalty

            sorted_logits, sorted_idx = torch.sort(next_token_logits, descending=True)
            sorted_probs = F.softmax(sorted_logits, dim=-1)

            cum_probs = torch.cumsum(sorted_probs, dim=-1)
            remove = cum_probs > topp
            remove[..., 0] = False            
            next_token_logits[0, sorted_idx[0, remove[0]]] = float('-inf')

            probs = F.softmax(next_token_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            
            idx = torch.cat((idx, next_token), dim=1)
            generated.append(next_token.item())

    text = tokenizer.decode(generated)
    return " ".join(text.split()[:128])

predictions = {str(id): {"text": continue_prompt(prompt["text"])} for id, prompt in test_data.items()}

In [ ]:
import zipfile

with open("prediction.json", "w", encoding="utf-8") as f:
    json.dump(predictions, f, ensure_ascii=False, indent=2)

with zipfile.ZipFile("submission.zip", "w", compression=zipfile.ZIP_DEFLATED) as zipf:
    zipf.write("prediction.json", arcname="prediction.json")